In [ ]:
import torchvision
import torch
from PIL import Image
from sklearn.metrics import confusion_matrix, accuracy_score
import torch.nn as nn
from torch import optim
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import re
import shutil
import random

In [19]:
# Define filtering function
def filter(input_folder, output_base_folder):
    # Make sure input folder exists
    if not os.path.isdir(input_folder):
        print(f"Error: {input_folder} is not a valid directory.")
        return

    # Create the output base folder if it doesn't exist
    os.makedirs(output_base_folder, exist_ok=True)

    # Iterate over each item in the input folder
    for item in os.listdir(input_folder):
        item_path = os.path.join(input_folder, item)
        if "patched_" in item_path or item_path[-3:] == 'tif' or os.path.isdir(item_path):
            pass
        else:
            size = os.stat(item_path).st_size
            if size > 2000 and bool(re.search("patch\d{1,3}", item_path)):
                shutil.copy(item_path, output_base_folder)
                print(f"Copied '{item}' to '{output_base_folder}'")

# call function - modify folder paths if needed !!!
if __name__ == "__main__":
    input_folder = 'Patches' 
    output_base_folder = "filtered_patches"

    filter(input_folder, output_base_folder)


Copied 'case_97_match_3_melan_patch32.png' to 'filtered_patches'
Copied 'case_78_match_1_melan_patch60.png' to 'filtered_patches'
Copied 'case_64_match_1_sox10_patch293.png' to 'filtered_patches'
Copied 'case_007_match_1_sox10_patch107.png' to 'filtered_patches'
Copied 'case_035_match_1_melan_patch19.png' to 'filtered_patches'
Copied 'case_84_match_1_sox10_patch63.png' to 'filtered_patches'
Copied 'case_059_match_1_sox10_patch326.png' to 'filtered_patches'
Copied 'case_79_match_3_melan_patch110.png' to 'filtered_patches'
Copied 'case_032_match_1_sox10_patch24.png' to 'filtered_patches'
Copied 'case_88_match_1_melan_patch25.png' to 'filtered_patches'
Copied 'case_72_match_1_h&e_patch21.png' to 'filtered_patches'
Copied 'case_97_match_2_melan_patch34.png' to 'filtered_patches'
Copied 'case_047_match_1_melan_patch42.png' to 'filtered_patches'
Copied 'case_72_match_1_h&e_patch35.png' to 'filtered_patches'
Copied 'case_88_match_1_melan_patch31.png' to 'filtered_patches'
Copied 'case_97_matc

In [29]:
# Replace last classifier to only handle 2 cases, one benign one high grade
num_classes = 2
model = torchvision.models.densenet121(weights=torchvision.models.DenseNet121_Weights.IMAGENET1K_V1)

# Modify the classifier
num_features = model.classifier.in_features
model.classifier = nn.Linear(num_features, num_classes)

# Freeze all layers except the classifier
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

labels = pd.read_csv('case_grade_match.csv')

In [30]:
def group_patches(patch_dir):
    case_patches = {}
    for filename in os.listdir(patch_dir):
        if os.path.getsize(os.path.join(patch_dir, filename)) < 2000:
            continue
        if 'patched_' in filename:
            continue
        elif filename.endswith('.png'):
            case_num = int(filename.split('_')[1])
            if case_num not in case_patches:
                case_patches[case_num] = []
            case_patches[case_num].append(os.path.join(patch_dir, filename))
    return case_patches

class PNGDataset(Dataset):
    def __init__(self, case_patches, labels_df, transform=None):
        self.case_patches = case_patches
        self.labels_df = labels_df
        self.transform = transform
        self.image_paths = []
        self.labels = []
        for case_num, patches in case_patches.items():
            label = labels_df.loc[labels_df['Case'] == case_num, 'Class'].values[0]
            label = 0 if label == 1 else 1
            for patch_path in patches:
                self.image_paths.append(patch_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

patches = group_patches('filtered_patches/')
case_nums = list(patches.keys())
dataset = labels.loc[[(int(x)-1) for x in case_nums]]
noindex = dataset.Class != 2.0
X = dataset[noindex].Case
y = dataset[noindex].Class
train, test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=40)

train_patches = {case_num: patches[int(case_num)] for case_num in train}
test_patches = {case_num: patches[int(case_num)] for case_num in test}

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = PNGDataset(train_patches, labels, transform=transform)
test_dataset = PNGDataset(test_patches, labels, transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)



In [31]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.0001)

num_epochs = 1000
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

patience = 0
best_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    epoch_loss = running_loss / len(train_dataloader)
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        patience = 0
    else:
        patience += 1
    
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss:.4f}, Patience: {patience}, Best Loss: {best_loss:.4f}')
    
    if patience >= 50:
        print(f'Early stopping at epoch {epoch + 1}')
        break


Epoch [1/1000], Loss: 0.6068, Patience: 0, Best Loss: 0.6068
Epoch [2/1000], Loss: 0.5806, Patience: 0, Best Loss: 0.5806
Epoch [3/1000], Loss: 0.5639, Patience: 0, Best Loss: 0.5639
Epoch [4/1000], Loss: 0.5631, Patience: 0, Best Loss: 0.5631
Epoch [5/1000], Loss: 0.5416, Patience: 0, Best Loss: 0.5416
Epoch [6/1000], Loss: 0.5332, Patience: 0, Best Loss: 0.5332
Epoch [7/1000], Loss: 0.5242, Patience: 0, Best Loss: 0.5242
Epoch [8/1000], Loss: 0.5238, Patience: 0, Best Loss: 0.5238
Epoch [9/1000], Loss: 0.5183, Patience: 0, Best Loss: 0.5183
Epoch [10/1000], Loss: 0.5162, Patience: 0, Best Loss: 0.5162
Epoch [11/1000], Loss: 0.5060, Patience: 0, Best Loss: 0.5060
Epoch [12/1000], Loss: 0.5087, Patience: 1, Best Loss: 0.5060
Epoch [13/1000], Loss: 0.5052, Patience: 0, Best Loss: 0.5052
Epoch [14/1000], Loss: 0.4932, Patience: 0, Best Loss: 0.4932
Epoch [15/1000], Loss: 0.4939, Patience: 1, Best Loss: 0.4932
Epoch [16/1000], Loss: 0.4915, Patience: 0, Best Loss: 0.4915
Epoch [17/1000], 

KeyboardInterrupt: 

In [32]:
pred = []
labels = []
with torch.no_grad():
    for images, label in test_dataloader:
        images, label = images.to(device), label.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        labels.append(label)
        pred.append(predicted)

pred = torch.cat(pred).cpu()
labels = torch.cat(labels).cpu()
accuracy = accuracy_score(labels, pred)
print(f'Accuracy: {accuracy}')
confusion_matrix(labels, pred)

Accuracy: 0.7407212885154062


array([[ 196,  796],
       [ 685, 4035]])